In [1]:

import os
import time
import subprocess
import webbrowser

# =====================================================================
# 1. WRITE THE STREAMLIT APP CODE AUTOMATICALLY TO app.py
# =====================================================================
app_code = """
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import streamlit as st

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

st.set_page_config(page_title="Customer Churn Analytics", layout="wide")

@st.cache_data
def load_and_generate_data():
    np.random.seed(42)
    n_samples = 1000
   
    data = {
        'CustomerID': [f"CUST-{1000+i}" for i in range(n_samples)],
        'Age': np.random.randint(18, 70, size=n_samples),
        'Tenure_Months': np.random.randint(1, 72, size=n_samples),
        'Monthly_Charges': np.round(np.random.uniform(20.0, 120.0, size=n_samples), 2),
        'Total_Transactions': np.random.randint(5, 150, size=n_samples),
        'Contract_Type': np.random.choice(['Month-to-Month', 'One-Year', 'Two-Year'], size=n_samples, p=[0.5, 0.3, 0.2]),
        'Payment_Method': np.random.choice(['Credit Card', 'Bank Transfer', 'Electronic Check'], size=n_samples),
        'Support_Tickets_Raised': np.random.randint(0, 10, size=n_samples)
    }
   
    df = pd.DataFrame(data)
    churn_score = (
        (df['Monthly_Charges'] / 120) * 0.35 +
        (df['Support_Tickets_Raised'] / 10) * 0.45 -
        (df['Tenure_Months'] / 72) * 0.4
    )
    df['Churn'] = (churn_score > 0.15).astype(int)
    return df

df = load_and_generate_data()

def train_ml_model(dataframe):
    df_proc = dataframe.copy()
    le_contract = LabelEncoder()
    le_payment = LabelEncoder()
   
    df_proc['Contract_Type'] = le_contract.fit_transform(df_proc['Contract_Type'])
    df_proc['Payment_Method'] = le_payment.fit_transform(df_proc['Payment_Method'])
   
    X = df_proc[['Age', 'Tenure_Months', 'Monthly_Charges', 'Total_Transactions',
                'Contract_Type', 'Payment_Method', 'Support_Tickets_Raised']]
    y = df_proc['Churn']
   
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
   
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
   
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)
   
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
   
    return model, scaler, le_contract, le_payment, acc, X_test, y_test, y_pred

model, scaler, le_contract, le_payment, accuracy, X_test, y_test, y_pred = train_ml_model(df)

st.title("📊 Customer Churn & Fraud Analytics Dashboard")
st.markdown("Predicting Customer Retention Risk using Machine Learning (Data Science Pipeline)")
st.divider()

col1, col2, col3, col4 = st.columns(4)
col1.metric("Total Customers", len(df))
col2.metric("Active Churn Rate", f"{(df['Churn'].mean()*100):.1f}%")
col3.metric("ML Model Accuracy", f"{accuracy*100:.2f}%")
col4.metric("Model Used", "Random Forest")
st.divider()

st.sidebar.header("🔍 Real-Time Churn Predictor")
input_age = st.sidebar.slider("Age", 18, 70, 35)
input_tenure = st.sidebar.slider("Tenure (Months)", 1, 72, 12)
input_charges = st.sidebar.number_input("Monthly Charges ($)", 20.0, 120.0, 75.0)
input_transactions = st.sidebar.number_input("Total Transactions", 1, 200, 30)
input_tickets = st.sidebar.slider("Support Tickets Raised", 0, 10, 3)
input_contract = st.sidebar.selectbox("Contract Type", ['Month-to-Month', 'One-Year', 'Two-Year'])
input_payment = st.sidebar.selectbox("Payment Method", ['Credit Card', 'Bank Transfer', 'Electronic Check'])

if st.sidebar.button("Predict Customer Risk"):
    contract_enc = le_contract.transform([input_contract])[0]
    payment_enc = le_payment.transform([input_payment])[0]
   
    user_data = np.array([[input_age, input_tenure, input_charges, input_transactions,
                           contract_enc, payment_enc, input_tickets]])
    user_data_scaled = scaler.transform(user_data)
   
    prediction = model.predict(user_data_scaled)[0]
    pred_prob = model.predict_proba(user_data_scaled)[0][1] * 100
   
    st.sidebar.divider()
    if prediction == 1:
        st.sidebar.error(f"⚠️ HIGH RISK: CHURN LIKELY\\nRisk Probability: {pred_prob:.2f}%")
    else:
        st.sidebar.success(f"✅ LOW RISK: RETENTION LIKELY\\nRetention Probability: {100-pred_prob:.2f}%")

tab1, tab2 = st.tabs(["📈 Exploratory Data Analysis (EDA)", "🤖 Model Evaluation & Metrics"])

with tab1:
    st.subheader("Customer Behavior Analysis")
    fig_col1, fig_col2 = st.columns(2)
    with fig_col1:
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.boxplot(x='Churn', y='Monthly_Charges', data=df, ax=ax, palette=['#2ecc71', '#e74c3c'])
        ax.set_xticklabels(['Retained (0)', 'Churned (1)'])
        ax.set_title("Monthly Charges vs Churn")
        st.pyplot(fig)
    with fig_col2:
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.barplot(x='Support_Tickets_Raised', y='Churn', data=df, ax=ax, color='#3498db')
        ax.set_title("Support Tickets Impact on Churn Risk")
        st.pyplot(fig)

with tab2:
    st.subheader("Machine Learning Performance")
    fig_col3, fig_col4 = st.columns(2)
    with fig_col3:
        st.write("**Confusion Matrix**")
        cm = confusion_matrix(y_test, y_pred)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Retained', 'Churned'], yticklabels=['Retained', 'Churned'])
        st.pyplot(fig)
    with fig_col4:
        st.write("**Feature Importance Ranking**")
        feature_names = ['Age', 'Tenure', 'Charges', 'Transactions', 'Contract', 'Payment', 'Tickets']
        importances = model.feature_importances_
        indices = np.argsort(importances)
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.barh(range(len(indices)), importances[indices], align='center', color='#9b59b6')
        ax.set_yticks(range(len(indices)))
        ax.set_yticklabels([feature_names[i] for i in indices])
        ax.set_xlabel('Relative Importance')
        st.pyplot(fig)
"""

# Save Code to app.py
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("✅ 'app.py' saved successfully!")

# =====================================================================
# 2. RUN STREAMLIT IN BACKGROUND & AUTO-OPEN BROWSER
# =====================================================================
# Start Streamlit process silently without blocking Jupyter
subprocess.Popen(["streamlit", "run", "app.py", "--server.headless=true"])

print("🚀 Launching Streamlit Server...")
time.sleep(2)  # Wait 2 seconds for server to boot

# Auto Open Dashboard in Chrome/Default Browser
dashboard_url = "http://localhost:8501"
webbrowser.open(dashboard_url)

print(f"🎉 SUCCESS! Streamlit App launched at: {dashboard_url}")

✅ 'app.py' saved successfully!
🚀 Launching Streamlit Server...
🎉 SUCCESS! Streamlit App launched at: http://localhost:8501
